In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Object-Oriented Design for Implementation

Minibatch stochastic gradient descent, as that section
described it, trains a model with a loop that is the same every time. The
loop runs for a number of *epochs*, each of them one pass over the training
data. In every epoch we draw minibatches, and for each minibatch we compute
the loss, its gradient, and an update of the parameters. After the epoch we
compute the loss on validation data, examples held out from training, to see
how well the model predicts examples it was not trained on. Written out, the
loop reads as follows.

```text
for epoch in range(num_epochs):
    for X, y in train_data:                     # data
        loss = loss_fn(net(X), y)               # model and loss
        update(params, gradient(loss, params))  # optimizer
    val_loss = mean_loss(net, val_data)         # model on held-out data
    record(epoch, val_loss)
```

Only the lines with comments depend on the problem; the loop around them
never changes. Nearly thirty sections of this book train a model, most of
them in several frameworks, so a loop written out in each would exist in
close to a hundred copies. A change to how we train, such as clipping very
large gradients or lowering the learning rate as training proceeds, would
then have to be made in every copy, and the copies would drift apart.

We therefore give the loop a class of its own, written once, and put the
parts that change into two more classes, a design borrowed from libraries
such as [Lightning](https://lightning.ai/):

* **`Module`** holds a model: its parameters, the `forward` computation of
  its predictions, its loss, and its *optimizer*, the object that applies an
  update rule such as minibatch SGD to the parameters. Every model in the
  book is a subclass.
* **`DataModule`** holds a dataset and serves its training and validation
  data in minibatches. Every dataset is a subclass.
* **`Trainer`** runs the loop. It takes minibatches from the `DataModule`,
  asks the `Module` for the loss on each, updates the parameters, and keeps a
  record of the losses.

In the loop above, the `DataModule` supplies `train_data` and `val_data`, the
`Module` supplies `net`, `loss_fn`, and the update rule, and the `Trainer`
runs the two `for` loops, computes the gradients, and records the losses.
the figure shows how the three work together. This section
writes the three base classes, and the next two sections fill them in for
linear regression: a dataset, then a model together with the two loops
inside the trainer.

![The three base classes. The `Trainer` takes minibatches from a `DataModule` and passes each to a `Module`, whose `training_step` returns the loss; the trainer updates the parameters and keeps the epoch means of the losses in its `history`. New models and datasets are written as subclasses of `Module` and `DataModule`.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-linreg-oo-classes.svg)

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
import collections

## Utilities

Two helpers make the classes of this book easier to write in a notebook. The
first concerns how the book presents code. Python defines a class in one
block of code, but a class with several methods is easier to explain in
several short cells, each next to the prose about it. The decorator
`add_to_class` registers a function as a method of a class that already
exists, so that one class definition can be spread over several cells. This
is notebook scaffolding, not a design we recommend: in an ordinary Python
module, a class is defined in one place.

In [2]:
def add_to_class(Class):
    """Register functions as methods in created class."""
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
        return obj
    return wrapper

`add_to_class(Class)` returns a decorator, and the decorator stores the
function as an attribute of `Class` under the function's own name. To see it
at work, we define a class `A` and create an instance `a` while `A` has no
methods other than its constructor.

In [3]:
class A:
    def __init__(self):
        self.b = 1

a = A()

In a separate cell we define a function `do` and register it with
`@add_to_class(A)`. Python looks up a method on the class at the moment it
is called, so `do` is available to the instance `a` that existed before
it. Inside `do`, `self` is that instance, and the method reads its attribute
`b` like a method written inside the class.

In [4]:
@add_to_class(A)
def do(self):
    print('Instance attribute "b" is', self.b)

a.do()

Instance attribute "b" is 1


The second helper removes repetitive code from constructors. A constructor
usually stores each of its arguments as an attribute: `self.lr = lr`,
`self.batch_size = batch_size`, and so on. These arguments are often
hyperparameters, settings such as the learning rate that we choose rather
than learn, hence the name of the class `HyperParameters`. Its method
`save_hyperparameters` stores all arguments of the constructor that calls
it.

In [5]:
class HyperParameters:
    """The base class of hyperparameters."""
    def save_hyperparameters(self, ignore=None):
        raise NotImplementedError

Here we only declare the method. Its implementation in that section
uses Python's `inspect` module to read the arguments of the method that
called it, and the `d2l` library holds the complete class. A class that
inherits from `d2l.HyperParameters` calls `save_hyperparameters` in its
constructor; arguments named in `ignore` are left out.

In [6]:
class B(d2l.HyperParameters):
    def __init__(self, a, b, c):
        self.save_hyperparameters(ignore=['c'])
        print('self.a =', self.a, 'self.b =', self.b)
        print('self.c exists:', hasattr(self, 'c'))

b = B(a=1, b=2, c=3)

self.a = 1 self.b = 2
self.c exists: False


All classes in the rest of this section inherit from `HyperParameters`, so
their constructors store their arguments with a single call.

## Models

A model in this book is a subclass of `Module`. The subclass creates its
parameters in its constructor and defines two methods: `forward`, which
computes the predictions for a minibatch of inputs, and `loss`, which
compares predictions with labels. (By default, `forward` applies `self.net`,
a network that the constructor of a subclass may store.) A third method,
`configure_optimizers`, returns the optimizer that updates the parameters;
for linear regression it is minibatch SGD with learning rate $\eta$.
`Module` inherits from `d2l.nn_Module`, the library's name for the base
class of networks in each framework, and from `HyperParameters`.

From `forward` and `loss`, the base class builds the two steps that the
trainer calls. A minibatch is a tuple whose last element holds the labels
and whose other elements hold the inputs. `training_step` therefore computes
`self.loss(self(*batch[:-1]), batch[-1])`: calling the model on the inputs
runs `forward`, and `loss` compares the predictions with the labels.
`validation_step` computes the same quantity on a minibatch of validation
data. Both steps *return* their result and do nothing else.

A step returns either the mean loss over the minibatch or a dictionary of
such minibatch means, for instance the loss and the accuracy of a classifier
in that section. A dictionary returned by `training_step`
must contain the key `'loss'`, the quantity that is differentiated. The
values must be means, not sums: to average over an epoch, the trainer
multiplies each value by the number of examples in its minibatch, so a loss
summed over a minibatch of 32 examples would enter the average 32 times too
large. Averaging over the epoch, recording, and plotting are left to the
`Trainer`.

In [7]:
class Module(d2l.nn_Module, d2l.HyperParameters):
    """The base class of models."""
    def __init__(self):
        super().__init__()
        self.training = None
        self.__dict__.pop('loss', None)

    def loss(self, y_hat, y):
        raise NotImplementedError

    def forward(self, X):
        assert hasattr(self, 'net'), 'Neural network is not defined'
        return self.net(X)

    def call(self, X, *args, training=None, **kwargs):
        if training is not None:
            self.training = training
        return self.forward(X, *args, **kwargs)

    def training_step(self, batch):
        """The loss on one batch, or a dict of metrics with key 'loss'."""
        return self.loss(self(*batch[:-1], training=True), batch[-1])

    def validation_step(self, batch):
        """The loss on one batch, or a dict of batch-mean metrics."""
        return self.loss(self(*batch[:-1], training=False), batch[-1])

    def configure_optimizers(self):
        raise NotImplementedError

`Module` is a subclass of `tf.keras.Model`, the base class of neural
networks in Keras. Calling an instance runs its method `call`, which we
redirect to `forward`, the name used throughout the book, passing on all
arguments except `training`. The training step
passes `training=True` and the validation step `training=False`, as Keras's
own training loop does. `call` stores this flag in `self.training`, so that
a forward computation that behaves differently during training, such as the
dropout written from scratch in a later chapter, can read it. The
constructor removes an attribute `loss` that Keras 3 sets to `None`, since
it would hide the method `loss` of our subclasses.

A step may return a single loss or a dictionary. The function `as_metrics`
turns either into a dictionary, so that the trainer handles both in the same
way.

In [8]:
def as_metrics(out):
    """A step returns a loss or a dict of metrics; return a dict."""
    return out if isinstance(out, dict) else {'loss': out}

## Data

A dataset in this book is a subclass of `DataModule`. Its constructor
prepares the data, downloading and preprocessing it if needed; the argument
`root` names the directory for downloaded files, and `num_workers` the number
of processes that load data in parallel. A subclass implements one method,
`get_dataloader(train)`. It returns a *data loader*, an object that yields
the data one minibatch at a time when we iterate over it, as the tuples that
the steps of `Module` expect. `train_dataloader` returns the loader for the
training data and `val_dataloader` the loader for the validation data.

In [9]:
class DataModule(d2l.HyperParameters):
    """The base class of data."""
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train):
        raise NotImplementedError

    def train_dataloader(self):
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        return self.get_dataloader(train=False)

## Training

### The Training Loop and the History

The `Trainer` class fits the parameters of a `Module` to the data of a
`DataModule`. Its constructor takes the number of epochs, `max_epochs`; the
arguments `num_gpus` and `gradient_clip_val` serve later chapters, which
train on GPUs and clip large gradients. The constructor also creates a
progress board, on which the trainer plots the losses during training; we
define it below. Training begins with three preparations, each a method
that later sections or frameworks override: `prepare_data` asks the
`DataModule` for its data loaders, `prepare_model` readies the model (later
it moves the model to a GPU), and `prepare_optimizer` calls the model's
`configure_optimizers`. The two loops over the data, `fit_epoch` for one
epoch of training and `validate` for one pass over the validation data,
depend on how each framework computes gradients. They are left abstract
here and written in that section.

In [10]:
class Trainer(d2l.HyperParameters):
    """Fit a Module to a DataModule, keeping a per-epoch history."""
    def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
        self.save_hyperparameters()
        self.board = ProgressBoard(xlabel='epoch', ylabel='loss')

    def prepare_data(self, data):
        self.train_dataloader = data.train_dataloader()
        self.val_dataloader = data.val_dataloader()

    def prepare_model(self, model):
        self.model = model

    def prepare_optimizer(self, model):
        self.optim = model.configure_optimizers()

    def fit_epoch(self):
        """Train for one epoch; return the mean of each training metric."""
        raise NotImplementedError

    def validate(self):
        """Return the mean of each metric over the validation set."""
        raise NotImplementedError

The method `fit` trains a `model`, an instance of `Module`, on `data`, an
instance of `DataModule`. After the three preparations it runs `max_epochs`
epochs. In each, `fit_epoch` updates the parameters once per minibatch and
returns the mean of every quantity that `training_step` reports, and
`validate` returns the means over the validation data. The method `record`
keeps these means.

In [11]:
@d2l.add_to_class(Trainer)
def fit(self, model, data):
    self.prepare_data(data)
    self.prepare_model(model)
    self.prepare_optimizer(model)
    self.history = collections.defaultdict(list)
    self.board.clear()
    self.board.xlim = [0, self.max_epochs]
    for self.epoch in range(self.max_epochs):
        # The training mean averages over the epoch, so it sits mid-epoch
        self.record('train', self.fit_epoch(), x=self.epoch + 0.5)
        if self.val_dataloader is not None:
            self.record('val', self.validate(), x=self.epoch + 1)
        self.board.show(final=False)
    self.board.show()

@d2l.add_to_class(Trainer)
def record(self, split, metrics, x):
    for name in sorted(metrics, key=lambda name: (name != 'loss', name)):
        self.history[f'{split}_{name}'].append(metrics[name])
        self.board.draw(x, metrics[name], f'{split}_{name}')

`fit` stores the epoch in the attribute `self.epoch`, so that the methods it
calls can read it. It skips validation when the `DataModule` has no
validation data, that is, when `val_dataloader` returns `None`. After every
epoch, `self.board.show(final=False)` redraws the board, and the final
`show()` draws it once more after training; the book build skips the
intermediate redraws.

`record` appends each mean to `trainer.history`, a dictionary of lists
whose keys combine the part of the data with the name of the quantity:
`'train_loss'`, `'val_loss'`, and, for a classifier, `'val_acc'`. `fit`
itself returns nothing. `record` takes the loss first and the other metrics
in alphabetical order, so that a curve keeps its color and line style on the
board from one model to the next, whatever order a step reports its metrics
in. After training, `trainer.history['val_loss'][-1]` is
the validation loss after the last epoch, and code that compares several
runs reads its results from `trainer.history`.

### Epoch Means

The means that `fit_epoch` and `validate` return are averages over
examples, not over minibatches: each minibatch counts in proportion to its
size. This matters for the last minibatch of a pass over the data, which is
smaller whenever the minibatch size does not divide the number of examples.
For instance, 1,000 examples in minibatches of 32 make 31 full minibatches
and one of 8. If the trainer averaged the 32 minibatch means with equal
weights, each of the last 8 examples would count four times as much as an
example in a full minibatch.

The model's steps return their means and the trainer averages and records
them, for two reasons. The first is that the model stays free of training
machinery. Lightning, whose design we otherwise follow, lets a step log its
own loss with `self.log`, which requires a reference to the trainer inside
the model. Here nothing in a model refers to the trainer or to the board.
Any loop, not only `fit`, can train or evaluate a model, and a model built
from the layers of a framework saves its parameters with that framework's
own tools (`state_dict` in PyTorch, `save_weights` in Keras, `nnx.state` in
Flax, `save_parameters` in Gluon).

The second reason is speed. A GPU runs *asynchronously*: Python places
operations in the device's queue and continues without waiting for them to
finish. Converting a loss into a Python number, to print or to plot it,
makes Python wait until the device has computed that loss, and the device
then idles until Python sends more work. A step that printed or plotted its
own loss would therefore make Python wait once per minibatch. A step that
returns its loss as an array on the device lets the trainer choose when to
convert: it adds up the returned losses on the device and converts the sums
into Python numbers once per epoch. that section shows this
accumulation in `fit_epoch`, and that section measures
what a conversion costs.

### The Progress Board

`record` also adds each mean to the progress board, which `fit` redraws
after every epoch so that we can watch training in a notebook. The class
`ProgressBoard` keeps one list of points per curve: `draw(x, y, label)`
appends the point `(x, y)` to the curve named `label`, `show` redraws all
curves, and `clear` forgets them. The drawing code is in that section.
The board belongs to the trainer and can be configured before training, for
instance with `trainer.board.yscale = 'log'` for a logarithmic loss axis, or
with `trainer.board.display = False` to train without plotting.

The two curves on the board are computed with different parameters. The
validation mean uses the parameters at the end of the epoch, while the
training mean averages over minibatches seen while the parameters changed
during the epoch. `fit` therefore draws the training mean of each epoch at
the middle of the epoch, at 0.5 for the first epoch, 1.5 for the second, and
so on, and the validation mean at the end of the epoch, at 1, 2, and so on.

In [12]:
class ProgressBoard(d2l.HyperParameters):
    """Curves that grow during training; `show` redraws them."""
    def __init__(self, xlabel=None, ylabel=None, xlim=None, ylim=None,
                 xscale='linear', yscale='linear', figsize=(3.5, 2.5),
                 display=True):
        self.save_hyperparameters()
        self.clear()

    def clear(self):
        """Forget all curves; the next `show` starts a new figure."""
        self.data, self.handle = {}, None

    def draw(self, x, y, label):
        """Append the point (x, y) to the curve called `label`."""
        xs, ys = self.data.setdefault(label, ([], []))
        xs.append(x)
        ys.append(y)

    def show(self, final=True):
        raise NotImplementedError

### Evaluation

Validation metrics are also needed outside `fit`, for instance to evaluate a
trained model once on a test set. `evaluate` asks `data` for its validation
loader only, readies the model, and returns the means of the metrics of
`model` over `data.val_dataloader()`. It sets the training loader to `None`,
so `data` needs no training data, and nothing in the evaluation draws a
training minibatch. The keys of the dictionary are those that
`validation_step` reports, without the prefix that `history` adds:
`{'loss': ...}` rather than `'val_loss'`.

In [13]:
@d2l.add_to_class(Trainer)
def evaluate(self, model, data):
    """Validation metrics of `model` on `data`, averaged over examples."""
    self.train_dataloader = None
    self.val_dataloader = data.val_dataloader()
    self.prepare_model(model)
    return self.validate()

### Optimizers and Schedules

Minibatch SGD with a constant learning rate does not settle at the
minimizer. It keeps fluctuating around it, and lowering the learning rate as
training proceeds, following a schedule, makes the fluctuations smaller
(the figure). Each framework attaches a schedule to the
optimizer in its own way.

In TensorFlow, a Keras schedule is passed to the optimizer as its learning
rate, so the default `prepare_optimizer` suffices. `prepare_model` also
compiles the training and validation steps with `tf.function`; we write it
together with the steps in that section.

We now have a training loop that every model in the book can use, but
nothing for it to run: no dataset, no model, and no body for the two loops.
The next section supplies a dataset whose true parameters we know, and
that section the linear regression model together with
`fit_epoch` and `validate`.

## Summary

Every model in this book is trained by the same loop, so we write it once
and keep it apart from what changes between models. A `Module` holds a
model, its loss, and its optimizer; a `DataModule` serves training and
validation data in minibatches; and a `Trainer` runs the loop. A model's
steps return the loss, or a dictionary of metrics, and do nothing else. The
trainer sums them on the device, averages them over the examples of each
epoch, keeps the epoch means in `trainer.history`, and draws them on its
progress board; `trainer.evaluate` computes the same means on the validation
loader of any `DataModule`. New models and
datasets are subclasses, written cell by cell with `add_to_class`, and a
change to training is made in one place.

## Exercises

1. [code] **Methods added later.** `add_to_class` stores a function on a
   class with `setattr(Class, obj.__name__, obj)`.
    1. Register a second method `do` on class `A` with a different body.
       Predict which version `a.do()` runs, then check.
    1. Attach a function `greet(self)` to the instance instead, with
       `setattr(a, 'greet', greet)`, and call `a.greet()`. Explain the
       error, and why the same function works when it is stored on the
       class.

1. [code] **Saving hyperparameters.** `save_hyperparameters` is
   implemented in that section with Python's `inspect` module, which
   gives it access to the local variables of the method that calls it.
    1. Insert the line `d = 4` before the call of `save_hyperparameters` in
       the constructor of `B`. Predict whether `self.d` exists after
       construction, then check.
    1. Implement a version of `save_hyperparameters` that receives the
       values to store as explicit arguments instead of reading the
       caller's local variables. Change `B` to use it, and verify that
       `self.a` and `self.b` exist and `self.c` does not.
    1. State one thing the `inspect`-based version provides that your
       version gives up, and one mistake that your version rules out.

1. **Optimizer and model.** The `Module` class keeps the optimizer in
   `configure_optimizers`, a *method of the model*, rather than passing it
   as an argument to `Trainer`. State the advantages of this design choice,
   and describe a training setup in which putting the optimizer on the
   model becomes awkward.

1. **Reading the training curves.** The trainer draws the mean training
   loss of each epoch at the middle of the epoch and the validation loss
   at its end.
    1. Early in training, the training curve often lies above the
       validation curve, although both sets of examples come from the same
       distribution. Explain why from the way the two means are computed.
    1. Suppose `fit` computed the training loss like the validation loss,
       with the parameters at the end of each epoch. What would this cost
       per epoch, and what would the gap between the two curves then
       measure?

1. [code] **Early stopping.** Training for a fixed number of epochs may
   stop too early or too late. Write a subclass of `d2l.Trainer` whose
   `fit` ends training once the validation loss has not decreased for
   `patience` consecutive epochs, deciding from `self.history` alone.
   Explain why no model or dataset class has to change. Once
   that section supplies `fit_epoch` and `validate`, test
   your trainer on the synthetic regression data with a large
   `max_epochs`.

1. [code] **Test split.** Once that section supplies
   `validate`, `trainer.evaluate(model, data)` returns the metrics of
   `model` over `data.val_dataloader()`. Write a subclass of
   `d2l.SyntheticRegressionData` whose constructor also draws a test set
   from the same distribution and whose `val_dataloader` serves it. Train
   a model on the original data, and evaluate it once on
   the test set after the last epoch. Explain why the test set is evaluated
   once at the end rather than after every epoch like the validation data,
   and what would go wrong if the epoch with the best test loss were
   reported. that section returns to this question.

1. [code] **Synchronizing every minibatch.** ● The `Trainer` adds up the
   minibatch losses on the device and converts them into Python numbers
   once per epoch. Once that section supplies `fit_epoch`,
   write a variant that calls `float()` on the loss of every minibatch and
   accumulates the Python numbers instead.
    1. Time one epoch of both versions on a GPU for linear regression with
       2 features and with about $10^4$ features, and explain the
       differences you measure with the asynchronous execution described
       in this section.
    1. Repeat the timing on a CPU. State under which conditions the
       per-minibatch conversion is measurably slower and when the two
       versions are indistinguishable.

[Discussions](https://d2l.discourse.group/t/6647)